In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS shopsphere.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE shopsphere.gold.daily_sales AS
WITH order_metrics AS (
    SELECT
        CAST(order_date AS DATE) AS sales_date,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(CASE WHEN order_status = 'Cancelled' THEN 1 END) AS cancelled_orders,
        COUNT(CASE WHEN order_status = 'Delivered' THEN 1 END) AS completed_orders,
        COUNT(CASE WHEN order_status = 'Returned' THEN 1 END) AS returned_orders
    FROM shopsphere.silver.orders
    GROUP BY sales_date
),
financial_metrics AS (
    SELECT
        CAST(o.order_date AS DATE) AS sales_date,
        SUM(oi.gross_amount) AS gross_sales,
        SUM(oi.discount_amount) AS discount_amount,
        SUM(CASE WHEN o.order_status = 'Returned' THEN oi.net_amount ELSE 0 END) AS returned_amount,
        SUM(CASE WHEN o.order_status NOT IN ('Cancelled', 'Returned') THEN oi.net_amount ELSE 0 END) AS net_sales_before_returns
    FROM shopsphere.silver.orders o
    INNER JOIN shopsphere.silver.order_items oi ON o.order_id = oi.order_id
    GROUP BY sales_date
)
SELECT
    om.sales_date,
    om.total_orders,
    om.completed_orders,
    om.cancelled_orders,
    om.returned_orders,
    COALESCE(fm.gross_sales, 0) AS gross_sales,
    COALESCE(fm.discount_amount, 0) AS discount_amount,
    COALESCE(fm.net_sales_before_returns - fm.returned_amount, 0) AS net_sales,
    CASE 
        WHEN om.total_orders > 0 THEN ROUND(COALESCE(fm.net_sales_before_returns - fm.returned_amount, 0) / om.total_orders, 2)
        ELSE 0 
    END AS average_order_value,
    CASE 
        WHEN om.total_orders > 0 THEN ROUND((om.cancelled_orders * 100.0) / om.total_orders, 2)
        ELSE 0 
    END AS cancellation_rate
FROM order_metrics om
LEFT JOIN financial_metrics fm ON om.sales_date = fm.sales_date
ORDER BY om.sales_date DESC;

In [0]:
%sql
-- Sample of daily sales with all metrics
SELECT 
  sales_date,
  total_orders,
  completed_orders,
  cancelled_orders,
  returned_orders,
  gross_sales,
  discount_amount,
  net_sales,
  average_order_value,
  cancellation_rate
FROM shopsphere.gold.daily_sales
ORDER BY sales_date DESC
LIMIT 20;